# 13.3 模型給整段回答多大機率？


模型逐步猜字，而偏好比較的是整段回答。若第一步正確片段機率0.8，接著第二步在前文已確定條件下也為0.8，整段機率是0.8×0.8=0.64。長段許多小數相乘容易變得很小；取log後變成相加，更方便計算。本節只把回答位置的log分數加起來。

前置是[W.4的機率與log](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.4)、[13.2答案遮罩](https://birdhackor.github.io/tiny-perceptron-vlm/13.2.html)與[1.14逐步生成](https://birdhackor.github.io/tiny-perceptron-vlm/1.14.html)。每一步條件機率都假設前面的目標文字已給定，這種訓練算分方式不是讓模型先隨機寫出一篇再評分。整段log機率採用sum加總；mean平均回答每步是另一種量，不能悄悄替換。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.alignment import sequence_log_probability

logits = torch.tensor([[[0.0, 2.0, 0.0], [2.0, 0.0, 0.0], [0.0, 0.0, 2.0]]])
labels = torch.tensor([[-100, 0, 2]])
score = sequence_log_probability(logits, labels)
print("整段log分數", round(score.item(), 4))
print("還原機率", round(score.exp().item(), 4))

輸入形狀(1,3,3)：1筆、3位置、每位置3個候選分數。第一位置標籤-100忽略，第二取候選0，第三取候選2；兩處正確候選分數都是2，其餘0。工具先做log_softmax把候選分數變成log機率，再按標籤取指定值，只加有效位置。

每處目標機率約0.7870，log約-0.2395，兩處相加輸出-0.4791；`exp()`反向還原，約0.6193，正是0.7870平方。第一位置盡管最高候選分數也為2，完全不計入整段回答。這確保prompt（提問前文）與填充位置不會替回答增加或減少分數。填充位置是把較短資料補到共同長度時加的占位格，不是實際回答；將其標籤設為-100，這個求和就忽略它。

更長回答多加一些非正的log機率，總分通常更負，這不直接表示質量差。如果新增第三個有效位置也有0.7870，sum會到約-0.7186，mean仍約-0.2395。後面的偏好訓練會拿一份參數保持不動的參考模型作基準：對同一個前文與回答，從訓練中的模型log分數減去參考模型log分數，再比較兩篇回答的差值。這裡先記住，不能單憑誰的絕對總分較高判人類偏好；基準比較會在[13.4](https://birdhackor.github.io/tiny-perceptron-vlm/13.4.html)展開。

每個目標機率是在對應前文條件下取得，因此0.64的例子不是假設兩字統計上彼此獨立。第一字可能影響第二字，模型在第二步已經看到了第一個目標字；連乘正好把這條條件鏈組合起來。本程式直接給三位置logits，沒有真的執行語言網路，目的是檢查選對格子與求和。換真實模型時，輸入與標籤必須依13.2的對齊方式，使每處logits來自正確前綴；即使算出的log數值有限，前綴若錯了也不是這篇答案的機率。結束標記若納入目標，也應同樣算進總分。

練習把labels第一項改1，讓原本忽略位置也成為有效答案。先預測三項各約-0.2395，總分約-0.7186、還原機率約0.4874，再執行核對。隨後解釋這為什麼只是增加一項的數學測試：若第一位置實際屬於prompt，它就不是回答內容，不能為了讓分數變化把它任意算進回答。真正的回答分數必須遵守原本的答案遮罩。
